# The Game

## Dependencies

In [170]:
from game import Game
import time
import os
import re
from collections import deque
import heapq
import math

## Load maps

In [171]:
def extract_map_files(directory):
    pattern = re.compile(r'^map(\d+)\.txt$')
    map_file_indices = []

    for file_name in os.listdir(directory):
        match = pattern.match(file_name)
        if match:
            map_file_indices.append(match.group(1))

    return [int(idx) for idx in map_file_indices]

def is_valid_input(map, indices, algorithm, solvers):
    valid_input = True
    if map not in indices:
        print(f"Map index out of range. Please choose within range {min(indices)} to {max(indices)}")
        valid_input = False
    if algorithm not in solvers.keys():    
        print(f"{algorithm} is not a defined algorithm. Please choose from", *[f"{solver} ({i+1})  " for i, solver in enumerate(solvers.keys())])
        valid_input = False
    return valid_input

def load_map(map_index):  
    file_name = "map" + str(map_index) + ".txt"
    with open('./assets/maps/' + file_name) as f:
        game_map = f.read()
    return game_map

map_file_indices = extract_map_files("./assets/maps/")

## Tutorial

In [172]:
print("This is an example of the game map:")
map = load_map(2)
game = Game(map)
game.display_map()

This is an example of the game map:
W	P1	H	W	W	W	W
W	W	W	G1	W	W	W
W	W	W	B1	W	W	W
W	G2	B2	.	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [173]:
game.get_box_locations()

[(2, 3), (3, 2), (4, 3)]

In [174]:
game.get_goal_locations()

[(1, 3), (3, 1), (5, 3)]

In [175]:
game.get_player_position()

(0, 2)

- W : Wall
- H : Human
- B : Box
- P : Portal
- G : Goal

In [176]:
for direction in ['U', 'D', 'R', 'L']:
    result = game.apply_move(direction)
    print(f"Move {direction} is valid: {result}")
    if result:
        game.display_map()

Move U is valid: False
Move D is valid: False
Move R is valid: False
Move L is valid: True
W	P1	.	W	W	W	W
W	W	W	G1	W	W	W
W	W	W	B1	W	W	W
W	G2	B2	H	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [177]:
game.apply_move('U')
game.display_map()

W	P1	.	W	W	W	W
W	W	W	G1/B1	W	W	W
W	W	W	H	W	W	W
W	G2	B2	.	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [178]:
game.apply_moves(['D', 'L', 'R', 'D']) 
game.display_map()
print("Is game won?", game.is_game_won())

W	P1	.	W	W	W	W
W	W	W	G1/B1	W	W	W
W	W	W	.	W	W	W
W	G2/B2	.	.	P1	W	W
W	W	W	H	W	W	W
W	W	W	G3/B3	W	W	W
W	W	W	W	W	W	W
Is game won? True


## Solvers

In this part we gonna solve and find a way to win the discribed game using uninformed search algorithms (BFS, DFS, IDS) and informed search algorithm (A*).


### Graph, Node, State

for searching we need a search tree or search graph based on our problem situation here I choosed `graph`, cause tree will cause infinit loop !, and also to search a graph we need nodes ! each node should present a state of the game so we should choose what we want to consider as `state`.

one approach is to define states as the `game object` or `game map`. it might be good idea but they take lots of space and make the process so slow which is not good. also they have too much unnecessary info, for example the position of walls we don't need them, as we have methods to change the positions in the game map and object, so when we visit each node can apply its positions to the game object. 

I defined my states as below:

We take the position of player (two numbers) and postion of boxes (several two numbers!) and concat them to produce a string which only shows the position of `player` and `boxes` the reason of concat them as a string is for improving search speed (strings can be founded faster than tupels).

but also we save the numeric form of player and boxes position in the node to apply them on game object. which is easier than decode the concated string. 

### BFS

We know that BFS search is an optimal solution which means it can find a way with minimum actions and also it is an uninformed search method so it will test all states without any priority to reach the goal.
for implementing BFS we use a qeueu (FIFO), that is our fringe also we have a `visited` and `future` set. visited is to keep the nodes that we visited already and future is to keep track of nodes we visited already or they are in our qeueu and we gonna visit them (cause we are searching a graph, not a tree, this makes our search faster as we've seen in slides) .

for each node in addition to positions we save its parent and the action that leads to this node and when we found a goal state we recursivly find the parents and actions and reverse them to find the solution.

rest of the code is the simple BFS algorithm and we return the founded path and size of visited nodes.

In [179]:
def solver_bfs(game_map):
    game = Game(game_map)
    init_player = game.get_player_position()
    init_boxes = game.get_box_locations()
    node = {'state': ''.join(str(num) for tup in ([init_player] + init_boxes) for num in tup),
            'player': init_player ,
            'boxes': init_boxes
            }
    if game.is_game_won():
        return [], 0
    fringe = deque()
    fringe.append(node)
    visited = set()
    future = set()

    while fringe:
        node = fringe.popleft()
        visited.add(node['state'])

        for direction in ['U', 'D', 'R', 'L']:
            game.set_player_position(node['player'])
            game.set_box_positions(node['boxes'])
            result = game.apply_move(direction)
            
            if result:
                boxes = game.get_box_locations()
                player =  game.get_player_position()
                child = {'state': ''.join(str(num) for tup in ([player] + boxes) for num in tup) ,
                         'player': player,
                         'boxes': boxes,
                         'action': direction ,
                         'parent': node}
                
                if game.is_game_won():
                    actions = []
                    while 'parent' in child:
                        actions.append(child['action'])
                        child = child['parent']
                    actions.reverse()
                    return actions, len(visited)
                
                if (child['state'] not in visited) and (child['state'] not in future) :
                    fringe.append(child)
                    future.add(child['state'])

    return None, len(visited)

### DFS

we know that DFS is a complete but not optimal approach, and also because of the depth searching approach for many difficult maps it wouldn't find a solution in a reasonable time, but we still use it because it uses really smaller amount of storage in compare to BFS, and if we have a non optimal solution it will find it so sooner than BFS find the optimal one, because DFS search less nodes to find non optimal one. so if we have easy small maps and we are not looking for omptimality and want speed of process we can use DFS

for dfs we take the same approach as BFS except that instead of queue i used stack (LIFO) additionaly we have to sets one of them is `global_visited` that shows number of visted nodes during the whole search process and another one is `visited` which keep track of visited nodes through a specific path search and is saved in each node, shows the visited nodes until current node in that specific path , we use visited set for our DFS search to avoid losing some nodes in graph that searched in a different path before.

this time instead of recursive approach, to create path actions when goal node was founded, we save the path up to here in the node it self results in removing recursive approach and parent field in each node. 

In [180]:
def solver_dfs(game_map):
    game = Game(game_map)
    init_player = game.get_player_position()
    init_boxes = game.get_box_locations()
    init_state = ''.join(str(num) for tup in ([init_player] + init_boxes) for num in tup)
    
    fringe = []

    initial_node = {
        'player': init_player,
        'boxes': init_boxes,
        'path': [],
        'visited': {init_state} 
    }
    fringe.append(initial_node)
    

    global_visited = {init_state}
    
    while fringe:
        node = fringe.pop()
        
        game.set_player_position(node['player'])
        game.set_box_positions(node['boxes'])
        
        if game.is_game_won():
            return node['path'], len(global_visited)
        
        for direction in ['U', 'D', 'R', 'L']:
            game.set_player_position(node['player'])
            game.set_box_positions(node['boxes'])
            
            if game.apply_move(direction):
                new_player = game.get_player_position()
                new_boxes = game.get_box_locations()
                child_state = ''.join(str(num) for tup in ([new_player] + new_boxes) for num in tup)
                
                if child_state not in node['visited']:
                    new_visited = node['visited'].copy()
                    new_visited.add(child_state)
                    child_node = {
                        'player': new_player,
                        'boxes': new_boxes,
                        'path': node['path'] + [direction],
                        'visited': new_visited
                    }
                    fringe.append(child_node)
                    global_visited.add(child_state)
    
    return None, len(global_visited)


### IDS

to use the advantages of BFS and DFS both, we use IDS , it has DFS maner that preforms on different depth levels, as it is like BFS so it generates optimal solution, but the time factor is better than BFS and is like DFS.

to implement this method i used recursive approach, we are in an always loop that each time increase the depth and run `recursive IDS` until finds the solution and breaks the loop, IDS recursivly visit nodes on a DFS manner so we need two visited set like previous part, instead of having local visited set for each path (and node) we have a single `visited` set that we remove unnecessary nodes from it. also we have a `global_visited` set like previous part.

as while loop continues until finds a solution for unsolvable maps like map 4 we have trouble and increase depth while we have only two nodes and search them each time without any result, so i defined a `previous_global_visited` set that takes the visited nodes for previous depth search operation if current and privous visited nodes were same it means we checked all the nodes and should exit loop and this way we wouldn't stuck on `unsolvable maps`.

In [181]:
def recursive_ids(game, node, depth, visited, global_visited):
    if game.is_game_won():
        return []
    
    if depth == 0:
        return None
    
    for direction in ['U', 'D', 'R', 'L']:
        game.set_player_position(node['player'])
        game.set_box_positions(node['boxes'])
        
        if game.apply_move(direction):
            new_player = game.get_player_position()
            new_boxes = game.get_box_locations()
            state = (new_player, tuple(new_boxes))
            
            if state in visited:
                continue
            
            visited.add(state)
            global_visited.add(state)
            
            child = {
                'state': state,
                'player': new_player,
                'boxes': new_boxes
            }
            result = recursive_ids(game, child, depth - 1, visited, global_visited)
            if result is not None:
                return [direction] + result
            
            visited.remove(state)
    
    return None

def solver_ids(game_map):
    depth = 0

    global_visited = set()
    previous_visited = set()

    while True:
        game = Game(game_map)
        init_player = game.get_player_position()
        init_boxes = game.get_box_locations()
        state = (init_player, tuple(init_boxes))
        root = {'state': state, 'player': init_player, 'boxes': init_boxes}
        
        visited = {state}
        global_visited.add(state)
        result = recursive_ids(game, root, depth, visited, global_visited)

        if global_visited == previous_visited:
            return None, len(global_visited)
        
        previous_visited = global_visited.copy()
        
        if result is not None:
            return result, len(global_visited)
        
        depth += 1


### A*

A* is our only informed search it check each node cost and heuristic and decide which node is the best to visit next if we only check heuristic our search would be inefficient because it is greedy and if we only check cost we would decrease speed of the search and our search is not informed any more

implementing this search method is exactly like BFS and DFS but for out fring instead of stack or queue we use a `priority queue` in this method as I explained we define cost as (cost + heuristic) so the priority queue give the cheaper node first to implement this queue I uesed heap. also when we talk about distance we mean manhattan distance

to define a heuristic for this problem one approach is to define it as below:
for each box find the `(distance of player to box + box to goal)` and then between costs of boxes find the one that is minimum it might be okay but we completely ignored the concept of portals in this heuristic so if there exists an optimal solution that uses portals we ignored it

so we define another better heuristic that consider portals:
for each box: first we find the distance of player to box without any portal then for each portal we find the distance of player to portal gate + portal second gate to box (we do this for each portal gate). and find the minimum distance between them then we repeat this process for distance of box to goal. at the end between costs of boxes we return minimum one. 

for weighted A* we just give a weight to heuristic part and cost became lighter so it becomes similar to greedy search and it might lost the optimal solution. 

In [182]:

def manhattan(p1,p2):
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

def heuristic(game):
    player = game.get_player_position()
    boxes = game.get_box_locations()
    goals = game.get_goal_locations()
    portals = game.get_portal_locations()
    result = []
    for i in range(len(boxes)):
        temp1 = manhattan(player, boxes[i])
        temp_portal = [math.inf]
        for portal in portals:
            temp2 = manhattan(player, portal[0]) + manhattan(portal[1], boxes[i])
            temp3= manhattan(player, portal[1]) + manhattan(portal[0], boxes[i])
            temp_portal.append(min(temp2, temp3))
        temp4 = min(temp1, min(temp_portal))

        temp5 = manhattan(boxes[i], goals[i])
        temp_portal2 = [math.inf]
        for portal in portals:
            temp6 = manhattan(boxes[i], portal[0]) + manhattan(portal[1], goals[i])
            temp7 = manhattan(boxes[i], portal[1]) + manhattan(portal[0], goals[i])
            temp_portal2.append(min(temp6, temp7))
        temp8 = min(temp5, min(temp_portal2))
        result.append(temp4 + temp8)
    return min(result)

def bad_heuristic(game):
    player = game.get_player_position()
    boxes = game.get_box_locations()
    goals = game.get_goal_locations()
    result = []
    for i in range(len(boxes)):
        temp = manhattan(player, boxes[i]) + manhattan(boxes[i], goals[i])
        result.append(temp)
    return min(result)

def solver_astar(game_map, heuristic_func=heuristic, weight=1):

    game = Game(game_map)
    init_player = game.get_player_position()
    init_boxes = game.get_box_locations()
    node = {'state': ''.join(str(num) for tup in ([init_player] + init_boxes) for num in tup),
            'player': init_player ,
            'boxes': init_boxes,
            'cost': 0
            }
    if game.is_game_won():
        return [], 0
    fringe = []
    counter = 0
    heapq.heappush(fringe, (0, counter, node))
    visited = set()
    future = set()

    while(True):
        if len(fringe) == 0:
            return None, len(visited)
        
        node = heapq.heappop(fringe)[2]
        visited.add(node['state'])

        game.set_player_position(node['player'])
        game.set_box_positions(node['boxes'])

        if game.is_game_won():
            actions = []
            while 'parent' in node:
                actions.append(node['action'])
                node = node['parent']
            actions.reverse()
            return actions, len(visited)

        for direction in ['U', 'D', 'R', 'L']:
            game.set_player_position(node['player'])
            game.set_box_positions(node['boxes'])
            result = game.apply_move(direction)
            
            if result:
                boxes = game.get_box_locations()
                player =  game.get_player_position()
                child = {'state': ''.join(str(num) for tup in ([player] + boxes) for num in tup) ,
                         'player': player,
                         'boxes': boxes,
                         'action': direction ,
                         'parent': node,
                         'cost': node['cost'] + 1
                         }
                
                if (child['state'] not in visited) and (child['state'] not in future) :
                    priority = child['cost'] + weight * heuristic_func(game)
                    counter += 1
                    heapq.heappush(fringe, (priority, -counter, child))
                    future.add(child['state'])

def solver_weighted_astar_2(game_map, heuristic_func=heuristic, weight=2):

    game = Game(game_map)
    init_player = game.get_player_position()
    init_boxes = game.get_box_locations()
    node = {'state': ''.join(str(num) for tup in ([init_player] + init_boxes) for num in tup),
            'player': init_player ,
            'boxes': init_boxes,
            'cost': 0
            }
    if game.is_game_won():
        return [], 0
    fringe = []
    counter = 0
    heapq.heappush(fringe, (0, counter, node))
    visited = set()
    future = set()

    while(True):
        if len(fringe) == 0:
            return None, len(visited)
        
        node = heapq.heappop(fringe)[2]
        visited.add(node['state'])

        game.set_player_position(node['player'])
        game.set_box_positions(node['boxes'])

        if game.is_game_won():
            actions = []
            while 'parent' in node:
                actions.append(node['action'])
                node = node['parent']
            actions.reverse()
            return actions, len(visited)

        for direction in ['U', 'D', 'R', 'L']:
            game.set_player_position(node['player'])
            game.set_box_positions(node['boxes'])
            result = game.apply_move(direction)
            
            if result:
                boxes = game.get_box_locations()
                player =  game.get_player_position()
                child = {'state': ''.join(str(num) for tup in ([player] + boxes) for num in tup) ,
                         'player': player,
                         'boxes': boxes,
                         'action': direction ,
                         'parent': node,
                         'cost': node['cost'] + 1
                         }
                
                if (child['state'] not in visited) and (child['state'] not in future) :
                    priority = child['cost'] + weight * heuristic_func(game)
                    counter += 1
                    heapq.heappush(fringe, (priority, -counter, child))
                    future.add(child['state'])

def solver_weighted_astar_6(game_map, heuristic_func=heuristic, weight=6):

    game = Game(game_map)
    init_player = game.get_player_position()
    init_boxes = game.get_box_locations()
    node = {'state': ''.join(str(num) for tup in ([init_player] + init_boxes) for num in tup),
            'player': init_player ,
            'boxes': init_boxes,
            'cost': 0
            }
    if game.is_game_won():
        return [], 0
    fringe = []
    counter = 0
    heapq.heappush(fringe, (0, counter, node))
    visited = set()
    future = set()

    while(True):
        if len(fringe) == 0:
            return None, len(visited)
        
        node = heapq.heappop(fringe)[2]
        visited.add(node['state'])

        game.set_player_position(node['player'])
        game.set_box_positions(node['boxes'])

        if game.is_game_won():
            actions = []
            while 'parent' in node:
                actions.append(node['action'])
                node = node['parent']
            actions.reverse()
            return actions, len(visited)

        for direction in ['U', 'D', 'R', 'L']:
            game.set_player_position(node['player'])
            game.set_box_positions(node['boxes'])
            result = game.apply_move(direction)
            
            if result:
                boxes = game.get_box_locations()
                player =  game.get_player_position()
                child = {'state': ''.join(str(num) for tup in ([player] + boxes) for num in tup) ,
                         'player': player,
                         'boxes': boxes,
                         'action': direction ,
                         'parent': node,
                         'cost': node['cost'] + 1
                         }
                
                if (child['state'] not in visited) and (child['state'] not in future) :
                    priority = child['cost'] + weight * heuristic_func(game)
                    counter += 1
                    heapq.heappush(fringe, (priority, -counter, child))
                    future.add(child['state'])

## Solve Function

In [183]:
SOLVERS = {
    "BFS": solver_bfs,
    "DFS": solver_dfs,
    "IDS": solver_ids,
    "A*": solver_astar,
    "WA*2": solver_weighted_astar_2 ,
    "WA*6":solver_weighted_astar_6
}

In [201]:
def solve(map, method):  
    
    if not is_valid_input(map, map_file_indices, method, SOLVERS):
        return
    
    file_name = "map" + str(map) + ".txt"
    with open('./assets/maps/' + file_name) as f:
        game_map = f.read()
    
    start_time = time.time()
    moves, numof_visited_states = SOLVERS[method](game_map)
    end_time = time.time()
    print(f"{method} took {round(end_time - start_time, 2)} seconds on map {map} and visited {numof_visited_states} states.")
    
    if moves is None:
        print("No Solution Found!")
    else:
        print(f"{len(moves)} moves were used: {moves}")

def solve_h(map, method, heu):  
    
    if not is_valid_input(map, map_file_indices, method, SOLVERS):
        return
    
    file_name = "map" + str(map) + ".txt"
    with open('./assets/maps/' + file_name) as f:
        game_map = f.read()
    
    start_time = time.time()
    moves, numof_visited_states = SOLVERS[method](game_map, heu)
    end_time = time.time()
    print(f"{method} took {round(end_time - start_time, 2)} seconds on map {map} and visited {numof_visited_states} states.")
    
    if moves is None:
        print("No Solution Found!")
    else:
        print(f"{len(moves)} moves were used: {moves}")
            

In [185]:
# def solve_all():
#     for map in range(min(map_file_indices), max(map_file_indices) + 1):
#         for method in SOLVERS.keys():
#             solve(map, method)

#solve_all() # Solve all maps using all methods

## Questions

### Q1

as I explained before, taking the whole map or game object as state is too big and has memory and speed problems also we would carry lots of data that we don't need, for example we don't need position of walls or portals (in uninformed searches) so it's better to extract the position of boxes and player from the map and take it as state in each node, I pesonally create a string that has mede of concatination of player and boxes position in order. (you can read more in `Graph, Node, State` section)

### Q2

as we take position of player and boxes as states the actions are player move in each direction it means in each state we have four action: left, right, up, down but all of these actions are not possible for all states as our game object can tell us one action can be performed or not we only apply those performable actions for each state. (you can read more in `Graph, Node, State` section)

### Q3

for initial state i consider position of player and boxes in input map (I don't change anything, take it as it is) and produce the explained string then for actions I use game object ability to check each action is performable or not and after action that means a new node we check that we win or not (we have this method in game object) so it means each state that the game object says we win is a goal state also to handle avoid of saving game object in each state, I used the game object method that sets boxes positions and player postion. 

### Q4

to avoid expandign all the four actions we have we use the game object ability that shows which moves can be done with a ture false result, we defently shouldn't expand all the four possible actions for each node because it increases the search graph nodes and make search progress slower.

### Q5

you can find the description of each algorithm above `its section` in this notebook and for the comparison we generally can say that DFS gives you the `optimal` answer but it search lots of useless nodes so it is not efficien, DFS is better than BFS in terms of speed but it is not trustable because it might stuck in infinite loop (not really loops but it might stuck in search of a useless path for a long time and result in time out) also DFS need less memory and space for fringe, IDS combine this two so it has their advantages and disadvanteges both, it finds `optimal` solution and uses little amount of space in comparison with BFS but sometimes it isn't trustable and sometimes depend on map situations it takes so much time to find the answer, A* is an informed search so it filters the nodes for searching and it is `optimal` also because of this filtering it visits less nodes than BFS and DFS so it is faster than those two, weighted A* is so much faster than A* because of the weight we give to heuristic but as the wieght increases, the A* become more similart to greedy search and lose the optimality option. optimal searches are BFS, IDS and A*

a. as I explained in the above text it sometimes stucks in a path so it is not trustable and takes so much time but we use it because it uses less space and memory in comparison with BFS and for simple maps it is so much faster.

b. IDS use DFS manner but it sets a limitation for depth (like BFS) in each itration in this way we can say it is a `limited DFS` so we use the advantages of efficiency in space usage of DFS and also we get the optimal solution because we have BFS manner by depth limitation.

### Q6
for this question look at `test section` and `solvers section`.


### Q7
to define a heuristic for this problem one approach is to define it as below:
for each box find the `(distance of player to box + box to goal)` and then between costs of boxes find the one that is minimum it might be okay but we completely ignored the concept of portals in this heuristic so if there exists an optimal solution that uses portals we ignored it

so we define another better heuristic that consider portals:
for each box: first we find the distance of player to box without any portal then for each portal we find the distance of player to portal gate + portal second gate to box (we do this for each portal gate). and find the minimum distance between them then we repeat this process for distance of box to goal. at the end between costs of boxes we return minimum one. 

as we ignore the portals in first heuristic we can't identify shortest distance and sometimes this heuritic is bigger that actual distance it means that this heuristic is not admissible, and also because of existence of portals concept this heuristic can't be consistent because the real cost of usage of portals is less than actual distance but we don't estimate it in this heuristic.

the second heuristic is admissible because it considers portals and this heuristic is always less than actual distance. but we can't claim it is completely consists, becuase sometimes portals affect on some blocks and their costs so it is consistent, but for other blocks because we consider boxes and goals as main idea and portals are not useful for these blocks we can't say it is consistent .


### Q8
for this question look at `heuristics` under `test section`.

### Q9
for this question look at `test section`.


## Test Analysis

**Do Not run this part if you have changed test map files.**

**This part code and text is for the current test maps set**

### Test 1

![./images/10.jpeg](./images/1.jpeg)

it is an easy test so all solvers can solve it also they all found an optimal solution.
we can see DFS visited less states than others and this is the benfit of DFS and because the map is so easy you can't see the befits of A* yet also IDS is not perform well because it is not BFS and it is not DFS 

In [186]:
map_num = 1
solve(map_num, "BFS")
solve(map_num, "DFS")
solve(map_num, "IDS")
solve(map_num, "A*")
solve(map_num, "WA*2")
solve(map_num, "WA*6")


BFS took 0.0 seconds on map 1 and visited 40 states.
7 moves were used: ['U', 'D', 'D', 'U', 'R', 'L', 'L']
DFS took 0.0 seconds on map 1 and visited 17 states.
7 moves were used: ['L', 'R', 'R', 'L', 'D', 'U', 'U']
IDS took 0.0 seconds on map 1 and visited 44 states.
7 moves were used: ['U', 'D', 'D', 'U', 'R', 'L', 'L']
A* took 0.0 seconds on map 1 and visited 41 states.
7 moves were used: ['D', 'U', 'L', 'R', 'U', 'D', 'R']
WA*2 took 0.0 seconds on map 1 and visited 41 states.
7 moves were used: ['D', 'U', 'L', 'R', 'U', 'D', 'R']
WA*6 took 0.0 seconds on map 1 and visited 41 states.
7 moves were used: ['D', 'U', 'L', 'R', 'U', 'D', 'R']


### Test 2

![./images/2.jpg](./images/2.jpg)

again, it's like test 1.

In [187]:
map_num = 2
solve(map_num, "BFS")
solve(map_num, "DFS")
solve(map_num, "IDS")
solve(map_num, "A*")
solve(map_num, "WA*2")
solve(map_num, "WA*6")

BFS took 0.0 seconds on map 2 and visited 18 states.
6 moves were used: ['L', 'U', 'D', 'D', 'U', 'L']
DFS took 0.0 seconds on map 2 and visited 13 states.
6 moves were used: ['L', 'L', 'R', 'D', 'U', 'U']
IDS took 0.0 seconds on map 2 and visited 22 states.
6 moves were used: ['L', 'U', 'D', 'D', 'U', 'L']
A* took 0.0 seconds on map 2 and visited 17 states.
6 moves were used: ['L', 'D', 'U', 'L', 'R', 'U']
WA*2 took 0.0 seconds on map 2 and visited 17 states.
6 moves were used: ['L', 'D', 'U', 'L', 'R', 'U']
WA*6 took 0.0 seconds on map 2 and visited 17 states.
6 moves were used: ['L', 'D', 'U', 'L', 'R', 'U']


### Test 3

![./images/3.jpg](./images/3.jpg)

in this test we can see that weighted A* is not optimal because it becomes similar to greedy search and ignore cost but we can see A* solvers visited so much less state than BFS and IDS but why it consumes more time because calculating heuristic takes time !

In [188]:
map_num = 3
solve(map_num, "BFS")
solve(map_num, "DFS")
solve(map_num, "IDS")
solve(map_num, "A*")
solve(map_num, "WA*2")
solve(map_num, "WA*6")

BFS took 0.0 seconds on map 3 and visited 111 states.
13 moves were used: ['U', 'L', 'D', 'D', 'U', 'U', 'U', 'U', 'R', 'D', 'D', 'D', 'D']
DFS took 0.0 seconds on map 3 and visited 47 states.
13 moves were used: ['U', 'L', 'D', 'D', 'U', 'U', 'U', 'U', 'R', 'D', 'D', 'D', 'D']
IDS took 0.01 seconds on map 3 and visited 122 states.
13 moves were used: ['U', 'L', 'D', 'D', 'U', 'U', 'U', 'U', 'R', 'D', 'D', 'D', 'D']
A* took 0.0 seconds on map 3 and visited 81 states.
13 moves were used: ['U', 'L', 'D', 'D', 'U', 'U', 'U', 'U', 'R', 'D', 'D', 'D', 'D']
WA*2 took 0.01 seconds on map 3 and visited 83 states.
14 moves were used: ['U', 'L', 'U', 'U', 'R', 'D', 'D', 'D', 'D', 'U', 'U', 'L', 'D', 'D']
WA*6 took 0.01 seconds on map 3 and visited 86 states.
14 moves were used: ['U', 'L', 'U', 'U', 'R', 'D', 'D', 'D', 'D', 'U', 'U', 'L', 'D', 'D']


### Test 4

this test is special and it has not a solution so all the solvers found this with same searched state count. in IDS section I explained about how i make IDS robust to this test.

In [189]:
map_num = 4
solve(map_num, "BFS")
solve(map_num, "DFS")
solve(map_num, "IDS")
solve(map_num, "A*")
solve(map_num, "WA*2")
solve(map_num, "WA*6")

BFS took 0.0 seconds on map 4 and visited 2 states.
No Solution Found!
DFS took 0.0 seconds on map 4 and visited 2 states.
No Solution Found!
IDS took 0.0 seconds on map 4 and visited 2 states.
No Solution Found!
A* took 0.0 seconds on map 4 and visited 2 states.
No Solution Found!
WA*2 took 0.0 seconds on map 4 and visited 2 states.
No Solution Found!
WA*6 took 0.0 seconds on map 4 and visited 2 states.
No Solution Found!


### Test 5

![./images/5.jpg](./images/5.jpg)

this map has less walls so the agent is more free to choose next state so the number of states increases and DFS can't solve it because it visits so many useless paths and it gets time out ! but IDS solves it

but we can see A* searches visited so much less nodes in comparison with BFS and IDS, as I explain before more weight result in more greedy manner that loses optimality but increase speed as you can see in results.

`I should mention something important here`:
we can define time as number of visited states or runtime sometimes these two have confilicts because running heuritics take time !!


In [191]:
map_num = 5
solve(map_num, "BFS")
solve(map_num, "IDS")
solve(map_num, "A*")
solve(map_num, "WA*2")
solve(map_num, "WA*6")

BFS took 0.07 seconds on map 5 and visited 4690 states.
15 moves were used: ['U', 'L', 'D', 'D', 'R', 'D', 'L', 'L', 'L', 'U', 'U', 'U', 'R', 'U', 'L']
IDS took 1.86 seconds on map 5 and visited 6250 states.
15 moves were used: ['U', 'L', 'D', 'D', 'R', 'D', 'L', 'L', 'L', 'U', 'U', 'U', 'R', 'U', 'L']
A* took 0.03 seconds on map 5 and visited 1812 states.
15 moves were used: ['L', 'U', 'L', 'D', 'L', 'R', 'D', 'R', 'D', 'L', 'L', 'U', 'L', 'U', 'U']
WA*2 took 0.01 seconds on map 5 and visited 562 states.
15 moves were used: ['L', 'U', 'L', 'D', 'L', 'R', 'D', 'R', 'D', 'L', 'L', 'U', 'L', 'U', 'U']
WA*6 took 0.0 seconds on map 5 and visited 216 states.
19 moves were used: ['L', 'D', 'L', 'L', 'U', 'U', 'R', 'D', 'D', 'R', 'D', 'L', 'L', 'U', 'U', 'U', 'R', 'U', 'L']


### Test 6

![./images/6.jpg](./images/6.jpg)

in this test we lost IDS here we need to explain about IDS as we visit visited nodes in each iteration IDS is usually memory efficient but we can't trust it to be time efficient.

here we can see A* searches are better than BFS in time efficiency but only simple A* would find optimal solution.


In [193]:
map_num = 6
solve(map_num, "BFS")
solve(map_num, "A*")
solve(map_num, "WA*2")
solve(map_num, "WA*6")

BFS took 0.25 seconds on map 6 and visited 14852 states.
34 moves were used: ['U', 'U', 'U', 'U', 'U', 'R', 'R', 'R', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'R', 'D', 'L', 'R', 'R', 'R', 'R', 'R', 'R', 'R']
A* took 0.16 seconds on map 6 and visited 8678 states.
34 moves were used: ['R', 'D', 'D', 'D', 'D', 'D', 'R', 'R', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'U', 'U', 'U', 'U', 'U', 'U', 'U', 'U', 'U', 'R', 'U', 'L', 'R', 'R', 'R', 'R', 'R', 'R', 'R']
WA*2 took 0.16 seconds on map 6 and visited 7686 states.
35 moves were used: ['U', 'U', 'U', 'L', 'L', 'L', 'U', 'U', 'L', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'L', 'D', 'R', 'L', 'L', 'L', 'L', 'L', 'L', 'L']
WA*6 took 0.06 seconds on map 6 and visited 3151 states.
36 moves were used: ['R', 'R', 'R', 'U', 'U', 'U', 'U', 'L', 'U', 'R', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'R', 'D', 'L', 'R', 'R', 'R', 'R', 'R', 'R', 

### Test 7

![./images/7.jpg](./images/7.jpg)

test 7 is complex test you can find it based on visited states, here you might see that A* visits so much less nodes in comparison to BFS but it takes more time to execute it is because ot the time we spend to calculate heuristic and preform priority queue. and again you can see weighted A* s are faster and 2 is optimal but 6 is not so 2 might be a better choice for default weight.


In [199]:
map_num = 7
solve(map_num, "BFS")
solve(map_num, "A*")
solve(map_num, "WA*2")
solve(map_num, "WA*6")

BFS took 8.45 seconds on map 7 and visited 503326 states.
34 moves were used: ['R', 'U', 'R', 'R', 'D', 'D', 'D', 'D', 'L', 'D', 'R', 'U', 'U', 'U', 'U', 'L', 'L', 'L', 'R', 'D', 'R', 'D', 'R', 'D', 'D', 'L', 'L', 'D', 'L', 'L', 'U', 'U', 'D', 'R']
A* took 12.58 seconds on map 7 and visited 435367 states.
34 moves were used: ['R', 'U', 'R', 'R', 'D', 'D', 'D', 'D', 'L', 'D', 'R', 'U', 'U', 'U', 'U', 'L', 'L', 'L', 'R', 'D', 'R', 'D', 'R', 'D', 'D', 'L', 'L', 'D', 'L', 'L', 'U', 'U', 'D', 'R']
WA*2 took 9.49 seconds on map 7 and visited 319350 states.
34 moves were used: ['R', 'U', 'R', 'R', 'D', 'D', 'D', 'D', 'L', 'D', 'R', 'U', 'U', 'U', 'U', 'L', 'L', 'L', 'R', 'D', 'R', 'D', 'R', 'D', 'D', 'L', 'L', 'D', 'L', 'L', 'U', 'U', 'D', 'R']
WA*6 took 4.67 seconds on map 7 and visited 160654 states.
38 moves were used: ['R', 'U', 'R', 'R', 'D', 'D', 'D', 'D', 'L', 'D', 'R', 'L', 'L', 'D', 'L', 'L', 'U', 'U', 'D', 'R', 'R', 'U', 'R', 'R', 'U', 'U', 'U', 'L', 'L', 'L', 'R', 'D', 'R', 'D', 'R

### Test 8

![./images/8.jpg](./images/8.jpg)

in this test we have confilict of runtime and visited nodes as I explained but the bold subject is WA*6 which is so bad it shows that how sometimes greedy manner cause a disaster!, also we checked this test in heuristics term so for analytics check heristic section


In [ ]:
map_num = 8
solve(map_num, "BFS")
solve(map_num, "IDS")
solve(map_num, "A*")
solve(map_num, "WA*2")
solve(map_num, "WA*6")

BFS took 0.07 seconds on map 8 and visited 5117 states.
14 moves were used: ['U', 'U', 'R', 'D', 'L', 'D', 'R', 'R', 'D', 'R', 'U', 'R', 'D', 'R']
IDS took 1.82 seconds on map 8 and visited 7493 states.
14 moves were used: ['U', 'U', 'R', 'D', 'L', 'D', 'R', 'R', 'D', 'R', 'U', 'R', 'D', 'R']
A* took 0.03 seconds on map 8 and visited 1083 states.
14 moves were used: ['U', 'U', 'R', 'D', 'L', 'D', 'R', 'R', 'D', 'R', 'U', 'R', 'D', 'R']
WA*2 took 0.04 seconds on map 8 and visited 1647 states.
17 moves were used: ['U', 'R', 'U', 'R', 'D', 'D', 'L', 'D', 'R', 'R', 'R', 'R', 'U', 'R', 'D', 'D', 'R']
WA*6 took 0.27 seconds on map 8 and visited 10373 states.
51 moves were used: ['U', 'R', 'D', 'R', 'R', 'U', 'R', 'R', 'D', 'L', 'L', 'U', 'L', 'D', 'L', 'L', 'L', 'U', 'R', 'U', 'U', 'L', 'D', 'D', 'U', 'L', 'D', 'R', 'D', 'L', 'D', 'D', 'L', 'D', 'R', 'R', 'R', 'U', 'U', 'U', 'L', 'U', 'R', 'R', 'L', 'D', 'D', 'L', 'L', 'U', 'R']


### Test 9

this is a really compelex test so none of my solvers can solve it !

i should say that it was a little dificult even for me !

In [ ]:
map_num = 9

### Test 10

![./images/10.jpg](./images/10.jpg)

this test is like test 5 and you can see everything is standard and ok. also all A* searches found optimal solution.


In [200]:
map_num = 10
solve(map_num, "BFS")
solve(map_num, "A*")
solve(map_num, "WA*2")
solve(map_num, "WA*6")

BFS took 2.78 seconds on map 10 and visited 227887 states.
46 moves were used: ['R', 'R', 'R', 'R', 'R', 'D', 'R', 'U', 'L', 'U', 'R', 'U', 'L', 'L', 'L', 'U', 'L', 'D', 'R', 'U', 'U', 'U', 'L', 'D', 'R', 'D', 'L', 'D', 'R', 'R', 'D', 'R', 'U', 'L', 'U', 'R', 'U', 'R', 'D', 'D', 'R', 'D', 'L', 'L', 'L', 'L']
A* took 2.63 seconds on map 10 and visited 113958 states.
46 moves were used: ['R', 'R', 'R', 'R', 'R', 'D', 'R', 'U', 'L', 'U', 'R', 'U', 'L', 'L', 'L', 'U', 'L', 'L', 'D', 'R', 'U', 'U', 'L', 'D', 'R', 'D', 'L', 'D', 'R', 'R', 'D', 'R', 'U', 'L', 'U', 'R', 'U', 'R', 'D', 'D', 'R', 'D', 'L', 'L', 'L', 'L']
WA*2 took 1.51 seconds on map 10 and visited 60812 states.
46 moves were used: ['R', 'R', 'R', 'R', 'R', 'D', 'R', 'U', 'L', 'U', 'R', 'U', 'L', 'L', 'L', 'U', 'L', 'L', 'D', 'R', 'U', 'U', 'L', 'D', 'R', 'D', 'L', 'D', 'R', 'R', 'D', 'R', 'U', 'L', 'U', 'R', 'U', 'R', 'D', 'D', 'R', 'D', 'L', 'L', 'L', 'L']
WA*6 took 0.28 seconds on map 10 and visited 10820 states.
46 moves wer

### A* heuristics

![./images/h.jpg](./images/h.jpg)

as you can see when we have portals in a map our old heuristic doesn't work properly becuase it is not admissible and this is result of ignoring portals !. we test these heuristics on map 8 because the concept of portals affects on shortest path.

In [202]:
solve_h(8,"A*", bad_heuristic)
solve_h(8,"A*", heuristic)

A* took 0.05 seconds on map 8 and visited 2461 states.
19 moves were used: ['U', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'U', 'R', 'D', 'D', 'R']
A* took 0.03 seconds on map 8 and visited 1083 states.
14 moves were used: ['U', 'U', 'R', 'D', 'L', 'D', 'R', 'R', 'D', 'R', 'U', 'R', 'D', 'R']
